In [1]:
import requests 
import mysql.connector 
import pandas as pd
import json
import re

In [2]:
# ==============================
# CONEXIÓN A MYSQL
# ==============================
conexion = mysql.connector.connect(
    host="localhost",
    user="root",          # <-- cambia si es necesario
    password="misql555"       # <-- cambia tu contraseña
)

cursor = conexion.cursor()

In [3]:
# ==============================
# CREAR BASE DE DATOS
# ==============================
cursor.execute("DROP DATABASE IF EXISTS rrhh_attrition")
cursor.execute("CREATE DATABASE IF NOT EXISTS rrhh_attrition")
cursor.execute("USE rrhh_attrition")

print("Base de datos creada o ya existente")

Base de datos creada o ya existente


In [4]:
# ==============================
# TABLA: departments
# ==============================
cursor.execute("""
CREATE TABLE IF NOT EXISTS departments (
    departments_id INT AUTO_INCREMENT PRIMARY KEY,
    department_name VARCHAR(100)
)
""")


# ==============================
# TABLA: job_roles
# ==============================
cursor.execute("""
CREATE TABLE IF NOT EXISTS job_roles (
    job_roles_id INT AUTO_INCREMENT PRIMARY KEY,
    role_name VARCHAR(100)
)
""")


# ==============================
# TABLA: employees
# ==============================
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    employees_id INT AUTO_INCREMENT PRIMARY KEY,
    employee_number INT,
    age TINYINT,
    gender ENUM('Male', 'Female'),
    marital_status ENUM('Single', 'Married', 'Unknown', 'Divorced'),
    education TINYINT,
    education_field VARCHAR(100),
    num_companies_worked TINYINT,
    total_working_years TINYINT
)
""")


# ==============================
# TABLA: employee_career_path
# ==============================
cursor.execute("""
CREATE TABLE IF NOT EXISTS employee_career_path (
    employee_career_path_id INT AUTO_INCREMENT PRIMARY KEY,
    employees_id INT,
    years_at_company TINYINT,
    years_in_current_role TINYINT,
    years_since_last_promotion TINYINT,
    years_with_curr_manager TINYINT,
    training_times_last_year TINYINT,
    FOREIGN KEY (employees_id) REFERENCES employees(employees_id)
)
""")


# ==============================
# TABLA: employee_finance_details
# ==============================
cursor.execute("""
CREATE TABLE IF NOT EXISTS employee_finance_details (
    employee_finance_details_id INT AUTO_INCREMENT PRIMARY KEY,
    employees_id INT,
    departments_id INT,
    job_roles_id INT,
    job_level TINYINT,
    business_travel ENUM('Non-travel', 'Travel_rarely', 'Travel_frequently'),
    distance_from_home INT,
    monthly_income DECIMAL(10,2),
    monthly_rate INT,
    daily_rate INT,
    hourly_rate INT,
    percent_salary_hike TINYINT,
    stock_option_level TINYINT,
    over_time TINYINT(1),
    FOREIGN KEY (employees_id) REFERENCES employees(employees_id),
    FOREIGN KEY (departments_id) REFERENCES departments(departments_id),
    FOREIGN KEY (job_roles_id) REFERENCES job_roles(job_roles_id)
)
""")


# ==============================
# TABLA: employee_surveys
# ==============================
cursor.execute("""
CREATE TABLE IF NOT EXISTS employee_surveys (
    employee_surveys_id INT AUTO_INCREMENT PRIMARY KEY,
    employees_id INT,
    attrition TINYINT(1),
    environment_satisfaction TINYINT,
    job_satisfaction TINYINT,
    relationship_satisfaction TINYINT,
    job_involvement TINYINT,
    performance_rating TINYINT,
    work_life_balance TINYINT,
    FOREIGN KEY (employees_id) REFERENCES employees(employees_id)
)
""")


print("Todas las tablas han sido creadas correctamente")


Todas las tablas han sido creadas correctamente


🧱 5. Insertar datos (IMPORTANTE: ORDEN)

Debes seguir este orden por las FK:

departments
job_roles
employees
employee_career_path
employee_finance_details
employee_surveys

In [5]:
df_abc = pd.read_csv("empleados_limpio.csv")

df_abc.head()

,age,attrition,business_travel,daily_rate,department,distance_from_home,education,education_field,employee_number,environment_satisfaction,...,performance_rating,relationship_satisfaction,stock_option_level,total_working_years,training_times_last_year,work_life_balance,years_at_company,years_in_current_role,years_since_last_promotion,years_with_curr_manager
0,41,True,travel_rarely,1102,sales,1,2,life_sciences,1,2,...,3,1,0,8,0.0,1,6,4,0,5.0
1,49,False,travel_frequently,279,research_&_development,8,1,life_sciences,2,3,...,4,4,1,10,3.0,3,10,7,1,7.0
2,37,True,travel_rarely,1373,research_&_development,2,2,other,4,4,...,3,2,0,7,3.0,3,0,0,0,0.0
3,33,False,travel_frequently,1392,research_&_development,3,4,life_sciences,5,4,...,3,3,0,8,3.0,3,8,7,3,0.0
4,27,False,travel_rarely,591,research_&_development,2,1,medical,7,1,...,3,4,1,6,3.0,3,2,2,2,2.0


🏢 6. Insertar departments

In [6]:
departments = df_abc['department'].unique()
print(departments)

for dep in departments:
    cursor.execute(
        "INSERT INTO departments (department_name) VALUES (%s)",
        (dep,)
    )

conexion.commit()
print('datos insertados')


['sales' 'research_&_development' 'human_resources']
datos insertados


💼 7. Insertar job_roles

In [7]:
roles = df_abc['job_role'].unique()
print(roles)

for role in roles:
    cursor.execute(
        "INSERT INTO job_roles (role_name) VALUES (%s)",
        (role,)
    )

conexion.commit()
print('datos insertados')

['sales_executive' 'research_scientist' 'laboratory_technician'
 'manufacturing_director' 'healthcare_representative' 'manager'
 'sales_representative' 'research_director' 'human_resources']
datos insertados


🧑‍💼 8. Insertar employees

In [8]:
# Definicion de columnas del dataset para insercion en mi tabla employees
cols = [
    'employee_number',
    'age',
    'gender',
    'marital_status',
    'education',
    'education_field',
    'num_companies_worked',
    'total_working_years'
]

# Creacion del dataset df_employees desde el original solo con las columnas definidas anteriormente
df_employees = df_abc[cols].copy()

# ==============Tratamiento de variables===========
# Capitalizar gender y marital_status
df_employees['gender'] = df_employees['gender'].str.capitalize()
df_employees['marital_status'] = df_employees['marital_status'].str.capitalize()

# Convertir valores vacíos o NaN a None (para mysql)
df_employees['total_working_years'] = df_employees['total_working_years'].apply(
    lambda x: int(x) if pd.notnull(x) else None
)
# ==============Fin de tratamiento==============

# Convertir DataFrame a lista de tuplas lista para mysql
data_employees = list(df_employees.itertuples(index=False, name=None))

# Creacion de la sentencia de insercion
query = """
INSERT INTO employees (
    employee_number,
    age,
    gender,
    marital_status,
    education,
    education_field,
    num_companies_worked,
    total_working_years
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

# insercion por bloque
cursor.executemany(query, data_employees)
conexion.commit()

print('datos insertados')

datos insertados


🧑‍💼 9. Insertar employee_career_path

In [9]:
# Obtener employees_id (clave para relaciones)
cursor.execute("SELECT employees_id, employee_number FROM employees")
rows = cursor.fetchall()

# Conviertes las filas del SElect en un dataframe
df_ids = pd.DataFrame(rows, columns=['employees_id', 'employee_number'])

# Combinar df_abc con df_ids para tener employees_id
df_career_path = df_abc.merge(df_ids, on='employee_number')

# SECCION DE INSERCION 

# Definicion de columnas del dataset para insercion en mi tabla 
cols = [
    'employees_id',
    'years_at_company',
    'years_in_current_role',
    'years_since_last_promotion',
    'years_with_curr_manager',
    'training_times_last_year'
]

df_career_path = df_career_path[cols]

# Convertir DataFrame a lista de tuplas lista para mysql
data_career_path = list(df_career_path.itertuples(index=False, name=None))

# Creacion de la sentencia de insercion
query = """
INSERT INTO employee_career_path (
    employees_id,
    years_at_company,
    years_in_current_role,
    years_since_last_promotion,
    years_with_curr_manager,
    training_times_last_year
) VALUES (%s, %s, %s, %s, %s, %s)
"""

# insercion por bloque
cursor.executemany(query, data_career_path)
conexion.commit()
print("Datos de employee_career_path insertados ✅")

Datos de employee_career_path insertados ✅


10. Insertar employee_finance_details

In [10]:
# 🔹 Paso 1: Obtener las claves foráneas desde MySQL
# Obtener employees_id
cursor.execute("SELECT employees_id, employee_number FROM employees")
rows_emp = cursor.fetchall()
df_emp_ids = pd.DataFrame(rows_emp, columns=['employees_id', 'employee_number'])

# Obtener departments_id
cursor.execute("SELECT departments_id, department_name FROM departments")
rows_dep = cursor.fetchall()
df_dep_ids = pd.DataFrame(rows_dep, columns=['departments_id', 'department'])

# Obtener job_roles_id
cursor.execute("SELECT job_roles_id, role_name FROM job_roles")
rows_roles = cursor.fetchall()
df_role_ids = pd.DataFrame(rows_roles, columns=['job_roles_id', 'job_role'])


# 🔹 Paso 2: Combinar con el dataset original
# Combinar df_abc con employees_id
df_finance = df_abc.merge(df_emp_ids, on='employee_number')

# Combinar con departments_id
df_finance = df_finance.merge(df_dep_ids, on='department')

# Combinar con job_roles_id
df_finance = df_finance.merge(df_role_ids, on='job_role')


# 🔹 Paso 3: Seleccionar solo las columnas necesarias
cols_finance = [
    'employees_id',
    'departments_id',
    'job_roles_id',
    'job_level',
    'business_travel',
    'distance_from_home',
    'monthly_income',
    'monthly_rate',
    'daily_rate',
    'hourly_rate',
    'percent_salary_hike',
    'stock_option_level',
    'over_time'
]

df_finance = df_finance[cols_finance]

# tratamiento
df_finance['business_travel'] = df_finance['business_travel'].str.capitalize()
df_finance.head(20)

# 🔹 Paso 4: Convertir a lista de tuplas para MySQL
data_finance = list(df_finance.itertuples(index=False, name=None))

# 🔹 Paso 5: Crear la sentencia SQL y ejecutar
query_finance = """
INSERT INTO employee_finance_details (
    employees_id,
    departments_id,
    job_roles_id,
    job_level,
    business_travel,
    distance_from_home,
    monthly_income,
    monthly_rate,
    daily_rate,
    hourly_rate,
    percent_salary_hike,
    stock_option_level,
    over_time
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

# Inserción por bloque
cursor.executemany(query_finance, data_finance)
conexion.commit()
print("Datos de employee_finance_details insertados ✅")

Datos de employee_finance_details insertados ✅


📈 11. Insertar employee_surveys

In [11]:
# 🔹 Paso 1: Obtener employees_id de la tabla employees
# Traer employee_number y employees_id
cursor.execute("SELECT employees_id, employee_number FROM employees")
rows_emp = cursor.fetchall()

# Convertir a DataFrame
df_emp_ids = pd.DataFrame(rows_emp, columns=['employees_id', 'employee_number'])

# 🔹 Paso 2: Combinar con el dataset original (df_abc)
# Merge para agregar employees_id a tu dataset
df_surveys = df_abc.merge(df_emp_ids, on='employee_number')

# 🔹 Paso 3: Seleccionar solo las columnas necesarias
cols = [
    'employees_id',
    'attrition',
    'environment_satisfaction',
    'job_satisfaction',
    'relationship_satisfaction',
    'job_involvement',
    'performance_rating',
    'work_life_balance'
]

df_surveys = df_surveys[cols]

# 🔹 Paso 4: Convertir a lista de tuplas para MySQL
data_surveys = list(df_surveys.itertuples(index=False, name=None))

# 🔹 Paso 5: Crear la sentencia SQL e insertar
query_surveys = """
INSERT INTO employee_surveys (
    employees_id,
    attrition,
    environment_satisfaction,
    job_satisfaction,
    relationship_satisfaction,
    job_involvement,
    performance_rating,
    work_life_balance
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

cursor.executemany(query_surveys, data_surveys)
conexion.commit()
print("Datos de employee_surveys insertados ✅")

Datos de employee_surveys insertados ✅


In [12]:
# ==============================
# CERRAR CONEXIÓN
# ==============================
cursor.close()
conexion.close()

print("Conexión cerrada")

Conexión cerrada
